# Prototype clear sar
Clear and filter sentinel data

In [1]:
import pandas as pd
import os 
import pyarrow.parquet as pq

# Set

In [2]:
year = 2024
raw_path = '../data/raw'
ds_name = f"s2_vessel_detections_{str(year)}.parquet"

inp = os.path.join(raw_path, ds_name)

out_path = '../data/interim'
out_name = f'cleaned_{ds_name}'
out = os.path.join(out_path, out_name)

# Import 

In [3]:
# pq.read_table -> get parquet data with the proper lib
# use_pandas_metadata=False -> ignore the pandas metadata
table = pq.read_table(inp, use_pandas_metadata=False)  
# to_pandas -> convert to pandas dataframe
# ignore_metadata=True -> ignore the pandas metadata
df = table.to_pandas(ignore_metadata=True)  

# Clean
Drop, rename, sort columns.

In [4]:
cols_to_drop = [
    'detect_timestamp', 
] 

df = df.drop(columns=cols_to_drop)

In [5]:
rename_dict = {
    'detect_lat': 'latitude',
    'detect_lon': 'longitude',
}

df = df.rename(columns=rename_dict)

In [6]:
cols_ordered = [
    'date', 'latitude', 'longitude', 'ssvid', 'speed_kn_inferred', 'heading_deg_inferred', 'length_m_inferred','presence_score', 'matching_score', 'cloud_score', 'likely_infrastructure'
]

df = df[cols_ordered]

In [7]:
df.head()

,date,latitude,longitude,ssvid,speed_kn_inferred,heading_deg_inferred,length_m_inferred,presence_score,matching_score,cloud_score,likely_infrastructure
0,2024-05-27,45.568029,13.716881,377221000,1.29,286.570150,104.3,0.748,5.228180,0.495234,False
1,2024-05-27,45.734934,13.633887,256566000,0.14,45.836624,284.5,0.901,0.009679,0.993159,False
2,2024-05-27,45.510128,13.570846,278763000,20.00,281.413530,39.1,0.948,0.000270,0.813599,False
3,2024-05-27,45.679020,13.609953,211483750,5.32,221.252961,19.4,0.640,0.000036,0.938132,False
4,2024-05-27,45.571595,13.697961,636021250,0.13,103.705361,232.2,0.921,0.011942,0.980036,False


# Filter fishing vessels

I chose the following filter:
- speed_kn_inferred: [0,9] (vessel ship at most in [2,8], and some at [0,8])
- lengh_m_inferred: <25 (usually at a maximum length of 18 m)
- presence_score: > 0.5
- cloud score: < 0.5 
- likely_infrascture: False 

In [8]:
n_total = len(df)

# Filters
spd_mask = df['speed_kn_inferred'] < 9
len_mask = df['length_m_inferred'] < 25
prs_mask = df['presence_score'] > 0.5
cld_mask = df['cloud_score'] < 0.5
inf_mask = df['likely_infrastructure'] == False

# Show how many elements are removed for each filter 
masks = {
    "speed < 9": spd_mask,
    "length < 25": len_mask,
    "presence > 0.5": prs_mask,
    "cloud < 0.5": cld_mask,
    "not infrastructure": inf_mask,
}

for name, mask in masks.items():
    n_pass = mask.sum()
    n_removed = n_total - n_pass
    perc_removed = 100 * n_removed / n_total
    print(f"{name:25s} removed {n_removed:8d} rows ({perc_removed:6.2f}%)")

# Perform the filter 
full_mask = spd_mask & len_mask & prs_mask & cld_mask & inf_mask
n_after = full_mask.sum()
print("\nCombined filters:")
print(f"  Records before filtering: {n_total}")
print(f"  Records after filtering:  {n_after}")
print(f"  Records removed:          {n_total - n_after} ({100*(n_total-n_after)/n_total:.2f}%)")

df = df.loc[full_mask].copy()


speed < 9                 removed     1033 rows (  8.33%)
length < 25               removed     7373 rows ( 59.47%)
presence > 0.5            removed     1149 rows (  9.27%)
cloud < 0.5               removed      915 rows (  7.38%)
not infrastructure        removed        5 rows (  0.04%)

Combined filters:
  Records before filtering: 12397
  Records after filtering:  4079
  Records removed:          8318 (67.10%)


# Clean no more useful columns 

In [9]:
cols_to_drop = [
    'matching_score',
    'cloud_score',
    'likely_infrastructure',
]

df = df.drop(columns=cols_to_drop)

# Export 

In [10]:
df.to_parquet(out, index=False)